In [12]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
import sys

import pandas as pd

sys.path.append("..")

from predql.converter import SConverter, TConverter

In [14]:
from relbench.datasets import Dataset
from relbench.datasets import get_dataset, get_dataset_names
from relbench.tasks import BaseTask
from relbench.tasks import get_task_names, get_task

get_dataset_names()

['rel-amazon',
 'rel-avito',
 'rel-event',
 'rel-f1',
 'rel-hm',
 'rel-stack',
 'rel-trial']

In [15]:
def get_timestamps(dataset             : Dataset, 
                   timedelta           : pd.Timedelta, 
                   num_eval_timestamps : int,
                   split               : str) -> "pd.Series[pd.Timestamp]":
    db = dataset.get_db(upto_test_timestamp=(split != "test"))

    corrective_timedelta = pd.Timedelta("1ms")
    
    if split == "train":
        start = dataset.val_timestamp - timedelta + corrective_timedelta
        end = db.min_timestamp + corrective_timedelta
        freq = -timedelta
    elif split == "val":
        start = dataset.val_timestamp + corrective_timedelta
        end = min(
            dataset.val_timestamp
            + timedelta * (num_eval_timestamps - 1),
            dataset.test_timestamp - timedelta,
            ) + corrective_timedelta
        freq = timedelta
    elif split == "test":
        start = dataset.test_timestamp + corrective_timedelta
        end = min(
            dataset.test_timestamp
            + timedelta * (num_eval_timestamps - 1),
            db.max_timestamp - timedelta,
            ) + corrective_timedelta
        freq = timedelta
    else:
        pass

    timestamps = pd.date_range(start=start, end=end, freq=freq)
    return timestamps

In [16]:
def process_df_rb(df_rb     : pd.DataFrame, 
                  fk        : str, 
                  timestamp : str, 
                  label     : str) -> pd.DataFrame:
    renamed_df_rb = df_rb.rename(columns={fk        : 'fk',
                                          timestamp : 'timestamp',
                                          label     : 'label'})
    df_rb = renamed_df_rb.sort_values(by=['timestamp', 'fk'])
    
    corrective_timedelta = pd.Timedelta("1ms")
    df_rb['timestamp'] = df_rb['timestamp'] + corrective_timedelta

    return df_rb

In [17]:
def merge_dataframes(df_rb     : pd.DataFrame, 
                     df_predql : pd.DataFrame) -> None:
    # if pd.api.types.is_bool_dtype(df_predql['label']):
    #     df_predql['label'] = df_predql['label'].astype(int)
    
    merged = pd.merge(
    df_rb,       
    df_predql,   
    on=['fk', 'timestamp', 'label'], 
    how='outer',       
    suffixes=('_rb', '_predql'), 
    indicator=True    
    )

    print(f"Only in RelBench:\n {merged[merged['_merge'] == 'left_only']}")
    print(f"Only in PredQL:\n {merged[merged['_merge'] == 'right_only']}")
    print(f"In both:\n {merged[merged['_merge'] == 'both']}")

In [18]:
def check_correctness(dataset            : Dataset, 
                      task               : BaseTask,
                      split              : str, 
                      predql_query       : str,
                      fk_col_name        : str,
                      timestamp_col_name : str,
                      label_col_name     : str) -> None:
    timestamps = get_timestamps(dataset, task.timedelta, task.num_eval_timestamps, split)
    
    print(f"TIMEDELTA: {task.timedelta}")
    print(f"NUM_EVAL_TIMESTAMPS: {task.num_eval_timestamps}")

    converter = TConverter(dataset.get_db(upto_test_timestamp=(split != "test")), timestamps)
    df_rb = task.get_table(split, mask_input_cols=False).df
    df_rb = process_df_rb(df_rb, fk_col_name, timestamp_col_name, label_col_name)
    df_predql = converter.convert(predql_query).df()
    
    print(f"------------------- START {split.upper()} -------------------")
    merge_dataframes(df_rb, df_predql)
    print(f"------------------- END {split.upper()} ---------------------")

# F1 Database

In [19]:
dataset_f1 = get_dataset(name="rel-f1", download=True)
get_task_names("rel-f1")

['driver-position', 'driver-dnf', 'driver-top3']

## Entity Classification Tasks

### driver-dnf

Task Description: For each driver predict the if they will DNF (did not finish) a race in the next 1 month. 

In [20]:
task_f1_dnf = get_task("rel-f1", "driver-dnf", download=False)

In [21]:
predql_query = """
    PREDICT MAX(results.statusId, 0, 30, DAYS) != 1
    FOR EACH drivers.driverId
    ASSUMING COUNT(results.*, -365, 0, DAYS) != 0
    WHERE MAX(results.statusId, 0, 30, DAYS) IS NOT NULL 
    ;
"""

In [22]:
# TRAIN

check_correctness(dataset=dataset_f1, 
                  task=task_f1_dnf, 
                  predql_query=predql_query,
                  split="train",
                  fk_col_name="driverId",
                  timestamp_col_name="date",
                  label_col_name="did_not_finish")

TIMEDELTA: 30 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40
------WHERE_START------
SELECT
    help.driverId AS fk,
    help.timestamp,
    help.label
FROM
    (
------HELP_PART_START------
    SELECT
        parent.driverId,
        predict.timestamp,
        predict.label
    FROM
        drivers parent
    JOIN
        (------ASSUMING_START------
    SELECT
        help.driverId AS fk,
        help.timestamp,
        help.label
    FROM
        (
    ------HELP_PART_START------
        SELECT
            parent.driverId,
            predict.timestamp,
            predict.label
        FROM
            drivers parent
        JOIN
            (------PREDICT_START------
        SELECT
            help.driverId AS fk,
            help.timestamp,
            CASE
                WHEN main.fk IS NOT NULL THEN TRUE
                ELSE FALSE
            END
         AS label
        FROM
            (
        ------HELP_PART_START------
            SELECT
                parent.driverId,
         

In [23]:
# VAL

check_correctness(dataset=dataset_f1, 
                  task=task_f1_dnf, 
                  predql_query=predql_query,
                  split="val",
                  fk_col_name="driverId",
                  timestamp_col_name="date",
                  label_col_name="did_not_finish")

TIMEDELTA: 30 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40
------WHERE_START------
SELECT
    help.driverId AS fk,
    help.timestamp,
    help.label
FROM
    (
------HELP_PART_START------
    SELECT
        parent.driverId,
        predict.timestamp,
        predict.label
    FROM
        drivers parent
    JOIN
        (------ASSUMING_START------
    SELECT
        help.driverId AS fk,
        help.timestamp,
        help.label
    FROM
        (
    ------HELP_PART_START------
        SELECT
            parent.driverId,
            predict.timestamp,
            predict.label
        FROM
            drivers parent
        JOIN
            (------PREDICT_START------
        SELECT
            help.driverId AS fk,
            help.timestamp,
            CASE
                WHEN main.fk IS NOT NULL THEN TRUE
                ELSE FALSE
            END
         AS label
        FROM
            (
        ------HELP_PART_START------
            SELECT
                parent.driverId,
         

In [24]:
# TEST

check_correctness(dataset=dataset_f1, 
                  task=task_f1_dnf, 
                  predql_query=predql_query,
                  split="test",
                  fk_col_name="driverId",
                  timestamp_col_name="date",
                  label_col_name="did_not_finish")

Loading Database object from /home/kolesiko/.cache/relbench/rel-f1/db...
Done in 0.04 seconds.
TIMEDELTA: 30 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40
------WHERE_START------
SELECT
    help.driverId AS fk,
    help.timestamp,
    help.label
FROM
    (
------HELP_PART_START------
    SELECT
        parent.driverId,
        predict.timestamp,
        predict.label
    FROM
        drivers parent
    JOIN
        (------ASSUMING_START------
    SELECT
        help.driverId AS fk,
        help.timestamp,
        help.label
    FROM
        (
    ------HELP_PART_START------
        SELECT
            parent.driverId,
            predict.timestamp,
            predict.label
        FROM
            drivers parent
        JOIN
            (------PREDICT_START------
        SELECT
            help.driverId AS fk,
            help.timestamp,
            CASE
                WHEN main.fk IS NOT NULL THEN TRUE
                ELSE FALSE
            END
         AS label
        FROM
            (
  

### driver-top3

Task Description: For each driver predict if they will qualify in the top-3 for a race in the next 1 month. 

In [25]:
task_f1_top3 = get_task("rel-f1", "driver-top3", download=False)

In [26]:
predql_query = """
    PREDICT MIN(qualifying.position, 0, 30, DAYS) <= 3
    FOR EACH drivers.driverId
    WHERE MIN(qualifying.position, 0, 30, DAYS) IS NOT NULL;
"""

In [27]:
# TRAIN

check_correctness(dataset=dataset_f1, 
                  task=task_f1_top3, 
                  predql_query=predql_query,
                  split="train",
                  fk_col_name="driverId",
                  timestamp_col_name="date",
                  label_col_name="qualifying")

TIMEDELTA: 30 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40
------WHERE_START------
SELECT
    help.driverId AS fk,
    help.timestamp,
    help.label
FROM
    (
------HELP_PART_START------
    SELECT
        parent.driverId,
        predict.timestamp,
        predict.label
    FROM
        drivers parent
    JOIN
        (------PREDICT_START------
    SELECT
        help.driverId AS fk,
        help.timestamp,
        CASE
            WHEN main.fk IS NOT NULL THEN TRUE
            ELSE FALSE
        END
     AS label
    FROM
        (
    ------HELP_PART_START------
        SELECT
            parent.driverId,
            for_each.timestamp
        FROM
            drivers parent
        JOIN
            (SELECT
            parent.driverId AS fk,
            time.timestamp AS timestamp
        FROM
            drivers parent
        CROSS JOIN
            timestamp_df time
        
    ) for_each
        ON
            for_each.fk = parent.driverId
    ------HELP_PART_END------
        ) help

In [28]:
# VAL

check_correctness(dataset=dataset_f1, 
                  task=task_f1_top3, 
                  predql_query=predql_query,
                  split="val",
                  fk_col_name="driverId",
                  timestamp_col_name="date",
                  label_col_name="qualifying")

TIMEDELTA: 30 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40
------WHERE_START------
SELECT
    help.driverId AS fk,
    help.timestamp,
    help.label
FROM
    (
------HELP_PART_START------
    SELECT
        parent.driverId,
        predict.timestamp,
        predict.label
    FROM
        drivers parent
    JOIN
        (------PREDICT_START------
    SELECT
        help.driverId AS fk,
        help.timestamp,
        CASE
            WHEN main.fk IS NOT NULL THEN TRUE
            ELSE FALSE
        END
     AS label
    FROM
        (
    ------HELP_PART_START------
        SELECT
            parent.driverId,
            for_each.timestamp
        FROM
            drivers parent
        JOIN
            (SELECT
            parent.driverId AS fk,
            time.timestamp AS timestamp
        FROM
            drivers parent
        CROSS JOIN
            timestamp_df time
        
    ) for_each
        ON
            for_each.fk = parent.driverId
    ------HELP_PART_END------
        ) help

In [29]:
# TEST

check_correctness(dataset=dataset_f1, 
                  task=task_f1_top3, 
                  predql_query=predql_query,
                  split="test",
                  fk_col_name="driverId",
                  timestamp_col_name="date",
                  label_col_name="qualifying")

TIMEDELTA: 30 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40
------WHERE_START------
SELECT
    help.driverId AS fk,
    help.timestamp,
    help.label
FROM
    (
------HELP_PART_START------
    SELECT
        parent.driverId,
        predict.timestamp,
        predict.label
    FROM
        drivers parent
    JOIN
        (------PREDICT_START------
    SELECT
        help.driverId AS fk,
        help.timestamp,
        CASE
            WHEN main.fk IS NOT NULL THEN TRUE
            ELSE FALSE
        END
     AS label
    FROM
        (
    ------HELP_PART_START------
        SELECT
            parent.driverId,
            for_each.timestamp
        FROM
            drivers parent
        JOIN
            (SELECT
            parent.driverId AS fk,
            time.timestamp AS timestamp
        FROM
            drivers parent
        CROSS JOIN
            timestamp_df time
        
    ) for_each
        ON
            for_each.fk = parent.driverId
    ------HELP_PART_END------
        ) help

## Entity Regression Tasks

### driver-position

Task Description: Predict the average finishing position of each driver all races in the next 2 months. 

In [30]:
task_f1_pos = get_task("rel-f1", "driver-position", download=False)

In [31]:
predql_query = """
    PREDICT AVG(results.positionOrder, 0, 60, DAYS)
    FOR EACH drivers.driverId
    WHERE AVG(results.positionOrder, 0, 60, DAYS) IS NOT NULL;
"""

In [32]:
# TRAIN

check_correctness(dataset=dataset_f1, 
                  task=task_f1_pos, 
                  predql_query=predql_query,
                  split="train",
                  fk_col_name="driverId",
                  timestamp_col_name="date",
                  label_col_name="position")

TIMEDELTA: 60 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40
------WHERE_START------
SELECT
    help.driverId AS fk,
    help.timestamp,
    help.label
FROM
    (
------HELP_PART_START------
    SELECT
        parent.driverId,
        predict.timestamp,
        predict.label
    FROM
        drivers parent
    JOIN
        (------PREDICT_START------
    SELECT
        help.driverId AS fk,
        help.timestamp,
        main.comp_col
     AS label
    FROM
        (
    ------HELP_PART_START------
        SELECT
            parent.driverId,
            for_each.timestamp
        FROM
            drivers parent
        JOIN
            (SELECT
            parent.driverId AS fk,
            time.timestamp AS timestamp
        FROM
            drivers parent
        CROSS JOIN
            timestamp_df time
        
    ) for_each
        ON
            for_each.fk = parent.driverId
    ------HELP_PART_END------
        ) help
    LEFT JOIN
        (------AGGREGATION_START------
        SELECT
    

In [33]:
# VAL

check_correctness(dataset=dataset_f1, 
                  task=task_f1_pos, 
                  predql_query=predql_query,
                  split="val",
                  fk_col_name="driverId",
                  timestamp_col_name="date",
                  label_col_name="position")

TIMEDELTA: 60 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40
------WHERE_START------
SELECT
    help.driverId AS fk,
    help.timestamp,
    help.label
FROM
    (
------HELP_PART_START------
    SELECT
        parent.driverId,
        predict.timestamp,
        predict.label
    FROM
        drivers parent
    JOIN
        (------PREDICT_START------
    SELECT
        help.driverId AS fk,
        help.timestamp,
        main.comp_col
     AS label
    FROM
        (
    ------HELP_PART_START------
        SELECT
            parent.driverId,
            for_each.timestamp
        FROM
            drivers parent
        JOIN
            (SELECT
            parent.driverId AS fk,
            time.timestamp AS timestamp
        FROM
            drivers parent
        CROSS JOIN
            timestamp_df time
        
    ) for_each
        ON
            for_each.fk = parent.driverId
    ------HELP_PART_END------
        ) help
    LEFT JOIN
        (------AGGREGATION_START------
        SELECT
    

In [34]:
# TEST

check_correctness(dataset=dataset_f1, 
                  task=task_f1_pos, 
                  predql_query=predql_query,
                  split="test",
                  fk_col_name="driverId",
                  timestamp_col_name="date",
                  label_col_name="position")

TIMEDELTA: 60 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40
------WHERE_START------
SELECT
    help.driverId AS fk,
    help.timestamp,
    help.label
FROM
    (
------HELP_PART_START------
    SELECT
        parent.driverId,
        predict.timestamp,
        predict.label
    FROM
        drivers parent
    JOIN
        (------PREDICT_START------
    SELECT
        help.driverId AS fk,
        help.timestamp,
        main.comp_col
     AS label
    FROM
        (
    ------HELP_PART_START------
        SELECT
            parent.driverId,
            for_each.timestamp
        FROM
            drivers parent
        JOIN
            (SELECT
            parent.driverId AS fk,
            time.timestamp AS timestamp
        FROM
            drivers parent
        CROSS JOIN
            timestamp_df time
        
    ) for_each
        ON
            for_each.fk = parent.driverId
    ------HELP_PART_END------
        ) help
    LEFT JOIN
        (------AGGREGATION_START------
        SELECT
    

# Stack-Exchange Q&A Website Database

In [35]:
dataset_stack = get_dataset(name="rel-stack", download=False)
get_task_names("rel-stack")

['user-engagement',
 'post-votes',
 'user-badge',
 'user-post-comment',
 'post-post-related']

## Entity Classification Tasks

### user-engagement

Task Description: For each user predict if a user will make any votes, posts, or comments in the next 3 months. 

In [36]:
task_stack_engage = get_task("rel-stack", "user-engagement", download=False)

In [37]:
predql_query = """
     PREDICT COUNT(votes.Id, 0, 91, DAYS) != 0
          OR COUNT(posts.Id, 0, 91, DAYS) != 0
          OR COUNT(comments.Id, 0, 91, DAYS) != 0
     FOR EACH users.Id
     ASSUMING COUNT(votes.Id, -20, 0, YEARS) != 0
           OR COUNT(posts.Id, -20, 0, YEARS) != 0
           OR COUNT(comments.Id, -20, 0, YEARS) != 0;
"""

In [38]:
# TRAIN
# must be not filtered
check_correctness(dataset=dataset_stack, 
                  task=task_stack_engage, 
                  predql_query=predql_query,
                  split="train",
                  fk_col_name="OwnerUserId",
                  timestamp_col_name="timestamp",
                  label_col_name="contribution")

Loading Database object from /home/kolesiko/.cache/relbench/rel-stack/db...
Done in 10.33 seconds.
TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
------ASSUMING_START------
SELECT
    help.Id AS fk,
    help.timestamp,
    help.label
FROM
    (
------HELP_PART_START------
    SELECT
        parent.Id,
        predict.timestamp,
        predict.label
    FROM
        users parent
    JOIN
        (------PREDICT_START------
    SELECT
        help.Id AS fk,
        help.timestamp,
        CASE
            WHEN main.fk IS NOT NULL THEN TRUE
            ELSE FALSE
        END
     AS label
    FROM
        (
    ------HELP_PART_START------
        SELECT
            parent.Id,
            for_each.timestamp
        FROM
            users parent
        JOIN
            (SELECT
            parent.Id AS fk,
            time.timestamp AS timestamp
        FROM
            users parent
        JOIN
            timestamp_df time
        ON
            time.timestamp >= parent.CreationDate
 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

------------------- START TRAIN -------------------
Only in RelBench:
                       timestamp      fk label     _merge
1249    2010-07-15 00:00:00.001      37     1  left_only
7790    2010-07-15 00:00:00.001     270     1  left_only
20439   2010-01-14 00:00:00.001     855     0  left_only
20440   2010-04-15 00:00:00.001     855     0  left_only
20441   2010-07-15 00:00:00.001     855     0  left_only
...                         ...     ...   ...        ...
1360845 2020-07-02 00:00:00.001  251501     0  left_only
1360846 2019-10-03 00:00:00.001  252540     0  left_only
1360847 2020-01-02 00:00:00.001  252540     0  left_only
1360848 2020-04-02 00:00:00.001  252540     0  left_only
1360849 2020-07-02 00:00:00.001  252540     0  left_only

[1613 rows x 4 columns]
Only in PredQL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
                       timestamp      fk label _merge
0       2010-10-14 00:00:00.001       0     0   both
1       2011-01-13 00

In [18]:
table_df = dataset_stack.get_db().table_dict['users'].df
table_df[table_df['Id'] == 37]

Loading Database object from /home/kolesiko/.cache/relbench/rel-stack/db...
Done in 7.62 seconds.


,Id,AccountId,DisplayName,Location,ProfileImageUrl,WebsiteUrl,AboutMe,CreationDate
37,37,9084.0,hadley,"Houston, TX",NaN,http://hadley.nz,I'm an assistant professor of Statistics at Ri...,2010-07-19 19:13:09.870


In [81]:
# VAL
# must be not filtered
check_correctness(dataset=dataset_stack, 
                  task=task_stack_engage, 
                  predql_query=predql_query,
                  split="val",
                  fk_col_name="OwnerUserId",
                  timestamp_col_name="timestamp",
                  label_col_name="contribution")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
------ASSUMING_START------
SELECT
    help.Id AS fk,
    help.timestamp,
    help.label
FROM
    (
------HELP_PART_START------
    SELECT
        parent.Id,
        predict.timestamp,
        predict.label
    FROM
        users parent
    JOIN
        (------PREDICT_START------
    SELECT
        help.Id AS fk,
        help.timestamp,
        CASE
            WHEN main.fk IS NOT NULL THEN TRUE
            ELSE FALSE
        END
     AS label
    FROM
        (
    ------HELP_PART_START------
        SELECT
            parent.Id,
            for_each.timestamp,
        FROM
            users parent
        JOIN
            (SELECT
            parent.Id AS fk,
            time.timestamp AS timestamp
        FROM
            users parent
        JOIN
            timestamp_df time
        ON
            time.timestamp >= parent.CreationDate
    ) for_each
        ON
            for_each.fk = parent.Id
    ------HELP_PART_END------
       

In [82]:
# TEST
# must be filtered
check_correctness(dataset=dataset_stack, 
                  task=task_stack_engage, 
                  predql_query=predql_query,
                  split="test",
                  fk_col_name="OwnerUserId",
                  timestamp_col_name="timestamp",
                  label_col_name="contribution")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
------ASSUMING_START------
SELECT
    help.Id AS fk,
    help.timestamp,
    help.label
FROM
    (
------HELP_PART_START------
    SELECT
        parent.Id,
        predict.timestamp,
        predict.label
    FROM
        users parent
    JOIN
        (------PREDICT_START------
    SELECT
        help.Id AS fk,
        help.timestamp,
        CASE
            WHEN main.fk IS NOT NULL THEN TRUE
            ELSE FALSE
        END
     AS label
    FROM
        (
    ------HELP_PART_START------
        SELECT
            parent.Id,
            for_each.timestamp,
        FROM
            users parent
        JOIN
            (SELECT
            parent.Id AS fk,
            time.timestamp AS timestamp
        FROM
            users parent
        JOIN
            timestamp_df time
        ON
            time.timestamp >= parent.CreationDate
    ) for_each
        ON
            for_each.fk = parent.Id
    ------HELP_PART_END------
       

### user-badge

Task Description: For each user predict if a user will receive a new badge in the next 3 months. 

In [39]:
task_stack_badge = get_task("rel-stack", "user-badge", download=False)

In [40]:
predql_query = """
    PREDICT COUNT(badges.Id, 0, 91, DAYS) != 0
    FOR EACH users.Id;
"""

In [41]:
# TRAIN

check_correctness(dataset=dataset_stack, 
                  task=task_stack_badge, 
                  predql_query=predql_query,
                  split="train",
                  fk_col_name="UserId",
                  timestamp_col_name="timestamp",
                  label_col_name="WillGetBadge")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
------PREDICT_START------
SELECT
    help.Id AS fk,
    help.timestamp,
    CASE
        WHEN main.fk IS NOT NULL THEN TRUE
        ELSE FALSE
    END
 AS label
FROM
    (
------HELP_PART_START------
    SELECT
        parent.Id,
        for_each.timestamp
    FROM
        users parent
    JOIN
        (SELECT
        parent.Id AS fk,
        time.timestamp AS timestamp
    FROM
        users parent
    JOIN
        timestamp_df time
    ON
        time.timestamp >= parent.CreationDate
) for_each
    ON
        for_each.fk = parent.Id
------HELP_PART_END------
    ) help
LEFT JOIN
    (------CONDITION_START------
    SELECT
        *
    FROM
        (------AGGREGATION_START------
        SELECT
            parent.Id AS fk,
            COUNT(aggr_tbl.Id)
         AS comp_col,
            time.timestamp AS timestamp
        FROM
            users parent
        CROSS JOIN
            timestamp_df time
        LEFT JOIN
            badge

In [42]:
# VAL

check_correctness(dataset=dataset_stack, 
                  task=task_stack_badge, 
                  predql_query=predql_query,
                  split="val",
                  fk_col_name="UserId",
                  timestamp_col_name="timestamp",
                  label_col_name="WillGetBadge")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
------PREDICT_START------
SELECT
    help.Id AS fk,
    help.timestamp,
    CASE
        WHEN main.fk IS NOT NULL THEN TRUE
        ELSE FALSE
    END
 AS label
FROM
    (
------HELP_PART_START------
    SELECT
        parent.Id,
        for_each.timestamp
    FROM
        users parent
    JOIN
        (SELECT
        parent.Id AS fk,
        time.timestamp AS timestamp
    FROM
        users parent
    JOIN
        timestamp_df time
    ON
        time.timestamp >= parent.CreationDate
) for_each
    ON
        for_each.fk = parent.Id
------HELP_PART_END------
    ) help
LEFT JOIN
    (------CONDITION_START------
    SELECT
        *
    FROM
        (------AGGREGATION_START------
        SELECT
            parent.Id AS fk,
            COUNT(aggr_tbl.Id)
         AS comp_col,
            time.timestamp AS timestamp
        FROM
            users parent
        CROSS JOIN
            timestamp_df time
        LEFT JOIN
            badge

In [43]:
# TEST

check_correctness(dataset=dataset_stack, 
                  task=task_stack_badge, 
                  predql_query=predql_query,
                  split="test",
                  fk_col_name="UserId",
                  timestamp_col_name="timestamp",
                  label_col_name="WillGetBadge")

Loading Database object from /home/kolesiko/.cache/relbench/rel-stack/db...
Done in 9.17 seconds.
TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
------PREDICT_START------
SELECT
    help.Id AS fk,
    help.timestamp,
    CASE
        WHEN main.fk IS NOT NULL THEN TRUE
        ELSE FALSE
    END
 AS label
FROM
    (
------HELP_PART_START------
    SELECT
        parent.Id,
        for_each.timestamp
    FROM
        users parent
    JOIN
        (SELECT
        parent.Id AS fk,
        time.timestamp AS timestamp
    FROM
        users parent
    JOIN
        timestamp_df time
    ON
        time.timestamp >= parent.CreationDate
) for_each
    ON
        for_each.fk = parent.Id
------HELP_PART_END------
    ) help
LEFT JOIN
    (------CONDITION_START------
    SELECT
        *
    FROM
        (------AGGREGATION_START------
        SELECT
            parent.Id AS fk,
            COUNT(aggr_tbl.Id)
         AS comp_col,
            time.timestamp AS timestamp
        FROM
           

## Entity Regression Tasks

### post-votes

Task Description: For each user post predict how many votes it will receive in the next 3 months 

In [33]:
task_stack_post_votes = get_task("rel-stack", "post-votes", download=False)

In [38]:
predql_query = """
    PREDICT COUNT_DISTINCT(votes.Id WHERE votes.votetypeid == 2, 0, 91, DAYS)
    FOR EACH posts.Id WHERE posts.PostTypeId == 1
                        AND posts.OwnerUserId IS NOT NULL
                        AND posts.OwnerUserId != -1
    ;
"""

In [43]:
# TRAIN

check_correctness(dataset=dataset_stack, 
                  task=task_stack_post_votes, 
                  predql_query=predql_query,
                  split="train",
                  fk_col_name="PostId",
                  timestamp_col_name="timestamp",
                  label_col_name="popularity")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
------PREDICT_START------
SELECT
    help.Id AS fk,
    help.timestamp,
    main.comp_col
 AS label
FROM
    (
------HELP_PART_START------
    SELECT
        parent.Id,
        for_each.timestamp
    FROM
        posts parent
    JOIN
        (SELECT
        parent.Id AS fk,
        time.timestamp AS timestamp
    FROM
        (SELECT
        *
    FROM
        posts unsorted_aggr_tbl
    JOIN
        (------EXPR_START------
    SELECT
        fk,
    FROM
        (------EXPR_START------
        SELECT
            fk,
        FROM
            (------CONDITION_START------
            SELECT
                *
            FROM
                (------ID_DOT_ID_START------
                SELECT
                    Id AS fk,
                    PostTypeId AS comp_col
                FROM
                    posts
                ------ID_DOT_ID_END------
            )
            WHERE
                comp_col == 1
            ------CONDITI

In [44]:
# VAL

check_correctness(dataset=dataset_stack, 
                  task=task_stack_post_votes, 
                  predql_query=predql_query,
                  split="val",
                  fk_col_name="PostId",
                  timestamp_col_name="timestamp",
                  label_col_name="popularity")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
------PREDICT_START------
SELECT
    help.Id AS fk,
    help.timestamp,
    main.comp_col
 AS label
FROM
    (
------HELP_PART_START------
    SELECT
        parent.Id,
        for_each.timestamp
    FROM
        posts parent
    JOIN
        (SELECT
        parent.Id AS fk,
        time.timestamp AS timestamp
    FROM
        (SELECT
        *
    FROM
        posts unsorted_aggr_tbl
    JOIN
        (------EXPR_START------
    SELECT
        fk,
    FROM
        (------EXPR_START------
        SELECT
            fk,
        FROM
            (------CONDITION_START------
            SELECT
                *
            FROM
                (------ID_DOT_ID_START------
                SELECT
                    Id AS fk,
                    PostTypeId AS comp_col
                FROM
                    posts
                ------ID_DOT_ID_END------
            )
            WHERE
                comp_col == 1
            ------CONDITI

In [45]:
# TEST

check_correctness(dataset=dataset_stack, 
                  task=task_stack_post_votes, 
                  predql_query=predql_query,
                  split="test",
                  fk_col_name="PostId",
                  timestamp_col_name="timestamp",
                  label_col_name="popularity")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
------PREDICT_START------
SELECT
    help.Id AS fk,
    help.timestamp,
    main.comp_col
 AS label
FROM
    (
------HELP_PART_START------
    SELECT
        parent.Id,
        for_each.timestamp
    FROM
        posts parent
    JOIN
        (SELECT
        parent.Id AS fk,
        time.timestamp AS timestamp
    FROM
        (SELECT
        *
    FROM
        posts unsorted_aggr_tbl
    JOIN
        (------EXPR_START------
    SELECT
        fk,
    FROM
        (------EXPR_START------
        SELECT
            fk,
        FROM
            (------CONDITION_START------
            SELECT
                *
            FROM
                (------ID_DOT_ID_START------
                SELECT
                    Id AS fk,
                    PostTypeId AS comp_col
                FROM
                    posts
                ------ID_DOT_ID_END------
            )
            WHERE
                comp_col == 1
            ------CONDITI

## Link Prediction Tasks

### user-post-comment

Task Description: Predict a list of existing posts that a user will comment in the next two months. 

In [23]:
task_stack_post_comm = get_task("rel-stack", "user-post-comment", download=False)

In [30]:
predql_query = """
    PREDICT LIST_DISTINCT(posts.Id WHERE posts.OwnerUserId != -1
                                     AND posts.OwnerUserId IS NOT NULL, 0, 91, DAYS)
    FOR EACH comments.UserId WHERE comments.UserId IS NOT NULL;
"""

In [31]:
# TRAIN
# must be not filtered
check_correctness(dataset=dataset_stack, 
                  task=task_stack_post_comm, 
                  predql_query=predql_query,
                  split="train",
                  fk_col_name="UserId",
                  timestamp_col_name="timestamp",
                  label_col_name="PostId")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
------PREDICT_START------
SELECT
    help.UserId AS fk,
    help.timestamp,
    main.comp_col
 AS label
FROM
    (
------HELP_PART_START------
    SELECT
        parent.UserId,
        for_each.timestamp
    FROM
        comments parent
    JOIN
        (SELECT
        parent.UserId AS fk,
        time.timestamp AS timestamp
    FROM
        (SELECT
        *
    FROM
        comments unsorted_aggr_tbl
    JOIN
        (------CONDITION_START------
    SELECT
        *
    FROM
        (------ID_DOT_ID_START------
        SELECT
            UserId AS fk,
            UserId AS comp_col
        FROM
            comments
        ------ID_DOT_ID_END------
    )
    WHERE
        comp_col IS NOT NULL
    ------CONDITION_END------
    ) expr
    ON
        unsorted_aggr_tbl.UserId = expr.fk
    ) parent
    JOIN
        timestamp_df time
    ON
        time.timestamp >= parent.CreationDate
) for_each
    ON
        for_each.fk = parent.UserId

ParserException: Parser Error: syntax error at or near "="

LINE 71:                 in_tbl. = parent.UserId
                                 ^

In [98]:
df_rb = process_df_rb(df_rb, fk='UserId', timestamp='timestamp', label='PostId')
df_predql['label'] = df_predql['label']

print(df_rb)
print(df_predql)

       timestamp      fk                                              label
5330  2011-03-31      11                                             [1581]
8423  2011-03-31      20                                             [2307]
3084  2011-03-31      23  [8178, 5035, 7671, 7886, 8051, 1314, 2164, 892...
15584 2011-03-31      48                                             [5350]
12342 2011-03-31      58                                             [8018]
...          ...     ...                                                ...
2121  2018-12-31  195003                                           [117672]
8722  2018-12-31  195118                                           [139512]
8415  2018-12-31  195144                                           [175920]
5833  2018-12-31  195340                                            [68022]
431   2018-12-31  195477                                           [190332]

[16152 rows x 3 columns]
             fk  timestamp                           label
0  

In [99]:
merge_dataframes(df_rb, df_predql)

Only in RelBench:
          timestamp      fk            label_rb label_predql     _merge
81812   2011-03-31    2559              [6315]          NaN  left_only
82216   2011-03-31    2572              [7652]          NaN  left_only
84790   2011-03-31    2655        [5114, 8099]          NaN  left_only
85132   2011-03-31    2666  [6620, 2716, 1420]          NaN  left_only
86900   2011-03-31    2723   [583, 7034, 5948]          NaN  left_only
...            ...     ...                 ...          ...        ...
2068375 2018-12-31  195003            [117672]          NaN  left_only
2068376 2018-12-31  195118            [139512]          NaN  left_only
2068377 2018-12-31  195144            [175920]          NaN  left_only
2068378 2018-12-31  195340             [68022]          NaN  left_only
2068379 2018-12-31  195477            [190332]          NaN  left_only

[2005 rows x 5 columns]
Only in PredQL:
          timestamp      fk label_rb  \
0       2011-03-31       0      NaN   
1       2

### post-post-related

Task Description: Predict a list of existing posts that users will link a given post to in the next two months.

In [123]:
# RelBench table

from relbench.tasks.stack import PostPostRelatedTask 

task_table_rb = PostPostRelatedTask(dataset_stack).make_table(db=db_stack, timestamps=timestamps_stack)
print(task_table_rb)

Table(df=
      timestamp  PostId postLinksIdList
0    2017-06-30   96362          [5578]
1    2017-06-30  156943        [111859]
2    2017-06-30  151830        [180081]
3    2017-06-30  139499  [10097, 37093]
4    2018-12-31  211907         [11654]
...         ...     ...             ...
4378 2016-03-31    2451         [31112]
4379 2018-06-30   46241         [26893]
4380 2014-06-30   27122         [53844]
4381 2017-12-31  209568         [13575]
4382 2018-12-31  114560          [2509]

[4383 rows x 3 columns],
  fkey_col_to_pkey_table={'PostId': 'posts', 'postLinksIdList': 'posts'},
  pkey_col=None,
  time_col=timestamp)


In [124]:
# PredQL table

# PredQL table
# NOTE: STATIC WHERE NEEDED

predql_query = """
    PREDICT LIST_DISTINCT(posts.Id, 1, 92, DAYS)
    FOR EACH users.Id;
"""

task_table_predql = tconverter_stack.convert(predql_query)
print(task_table_predql)

------PREDICT_START------
SELECT
    parent.Id AS fk,
    time.timestamp AS timestamp,
    main.comp_col
 AS label
FROM
    users parent
CROSS JOIN
    timestamp_df time
LEFT JOIN
    (------AGGREGATION_START------
    SELECT
        parent.Id AS fk,
        (
        SELECT
            ARRAY_AGG(freq_tbl.val ORDER BY freq_tbl.freq DESC)
        FROM (
            SELECT
                in_tbl.Id AS val,
                COUNT(*) AS freq
            FROM
                posts in_tbl
            WHERE
                in_tbl.CreationDate >= time.timestamp + INTERVAL '1 DAY'
            AND
                in_tbl.CreationDate <  time.timestamp + INTERVAL '92 DAY'
            AND
                in_tbl.OwnerUserId = parent.Id
            GROUP BY in_tbl.Id
            ) freq_tbl
        )
     AS comp_col,
        time.timestamp AS timestamp
    FROM
        users parent
    CROSS JOIN
        timestamp_df time
    LEFT JOIN
        posts aggr_tbl
    ON
        aggr_tbl.OwnerUserId = paren

In [125]:
df_rb = task_table_rb.df
df_predql = task_table_predql.df()

In [126]:
df_rb = process_df_rb(df_rb, fk='PostId', timestamp='timestamp', label='postLinksIdList')

print(df_rb)
print(df_predql)

      timestamp      fk                          label
3130 2011-03-31      88                         [1574]
2283 2011-03-31     211                          [525]
1056 2011-03-31    2532                         [8069]
3250 2011-03-31    5236                         [6056]
2199 2011-03-31    5961                         [8166]
...         ...     ...                            ...
4168 2018-12-31  255382                        [59811]
461  2018-12-31  255385                 [78672, 53906]
2073 2018-12-31  255389                [122362, 93211]
3341 2018-12-31  255401                       [173128]
3307 2018-12-31  255418  [22174, 87268, 18138, 176509]

[4383 rows x 3 columns]
             fk  timestamp                           label
0             0 2011-03-31  [8976, 8974, 9221, 8973, 9222]
1             1 2011-03-31                            <NA>
2             2 2011-03-31                            <NA>
3             3 2011-03-31                            <NA>
4             4 2011

In [127]:
merge_dataframes(df_rb, df_predql)

Only in RelBench:
          timestamp      fk                       label_rb label_predql  \
8171520 2018-12-31  255382                        [59811]          NaN   
8171521 2018-12-31  255385                 [78672, 53906]          NaN   
8171522 2018-12-31  255389                [122362, 93211]          NaN   
8171523 2018-12-31  255401                       [173128]          NaN   
8171524 2018-12-31  255418  [22174, 87268, 18138, 176509]          NaN   

            _merge  
8171520  left_only  
8171521  left_only  
8171522  left_only  
8171523  left_only  
8171524  left_only  
Only in PredQL:
          timestamp      fk label_rb  \
0       2011-03-31       0      NaN   
1       2011-06-30       0      NaN   
2       2011-09-30       0      NaN   
3       2011-12-31       0      NaN   
4       2012-03-31       0      NaN   
...            ...     ...      ...   
8171515 2017-12-31  255359      NaN   
8171516 2018-03-31  255359      NaN   
8171517 2018-06-30  255359      NaN   
817